In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

feature_cols = [col for col in df.columns if col not in ["date", "sku_id", "units_sold"]]

# models to compare
model_configs = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
}

all_results = []
trained_models = {}  # trained_models[model_name][sku_id] = model object

for model_name, model_template in model_configs.items():
    trained_models[model_name] = {}
    
    for sku in df["sku_id"].unique():
        sku_df = df[df["sku_id"] == sku].sort_values("date").reset_index(drop=True)
        
        split_idx = int(len(sku_df) * 0.8)
        train = sku_df.iloc[:split_idx]
        test = sku_df.iloc[split_idx:]
        
        X_train, y_train = train[feature_cols], train["units_sold"]
        X_test, y_test = test[feature_cols], test["units_sold"]
        
        model = model_template.__class__(**model_template.get_params())  # fresh copy
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
        r2 = r2_score(y_test, preds)
        
        all_results.append({
            "model": model_name,
            "sku_id": sku,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2
        })
        trained_models[model_name][sku] = model

results_df = pd.DataFrame(all_results)
results_df